# gLM2 Genomic Language Model

![gLM2](https://proto-bio.github.io/proto-assets/images/tool/glm2/hero.png)

This notebook demonstrates the four functions of the gLM2 tool. In the first section, we use `run_glm2_embeddings` to extract a locus embedding and per-token logits. In the second section, we use `run_glm2_score` to compute masked pseudo-log-likelihood metrics that measure how natural a locus appears to the model. In the third section, we use `run_glm2_sample` to resample masked positions while keeping each position's protein or DNA identity. In the fourth section, we use `run_glm2_gradient` to compute gradients of the masked log-likelihood with respect to a relaxed sequence. Unlike protein-only language models, gLM2 reads a genomic locus as a mix of protein-coding genes and intergenic DNA, so its representations capture context from neighboring genes and the regulatory sequence between them.

A locus is written with uppercase amino acids for each gene, lowercase `acgt` for intergenic DNA, and a `+` or `-` strand marker before each element.

In [1]:
from pathlib import Path
from tempfile import mkdtemp

from proto_tools.tools.masked_models.glm2 import (
    GLM2EmbeddingsConfig, GLM2EmbeddingsInput, run_glm2_embeddings,
    GLM2ScoringConfig, GLM2ScoringInput, run_glm2_score,
    GLM2SampleConfig, GLM2SampleInput, run_glm2_sample,
    GLM2GradientConfig, GLM2GradientInput, run_glm2_gradient,
)
from proto_tools.tools.masked_models.mixed_data_models import one_hot_mixed_logits
from proto_tools.transforms.masking import MaskingStrategy
from proto_tools.utils.notebook_docs import display_api_reference, display_available_tools

CHECKPOINT = "tattabio/gLM2_150M"
DEVICE = "cuda"
OUTPUT_DIR = Path(mkdtemp(prefix="proto_glm2_example_"))

display_available_tools("glm2")

- **`run_glm2_embeddings()`** — Extract mixed protein/DNA embeddings and biological logits with gLM2
- **`run_glm2_gradient()`** — Differentiate masked pseudo-log-likelihood for relaxed mixed protein/DNA tokens with gLM2
- **`run_glm2_sample()`** — Sample mixed protein/DNA loci while preserving modality and strand markers with gLM2
- **`run_glm2_score()`** — Score mixed protein/DNA loci by masked pseudo-log-likelihood with gLM2

## Prepared example locus

The protein and DNA segments below are shortened from the mixed-locus example in the [gLM2 upstream README](https://github.com/TattaBio/gLM2#usage). This illustrates the input notation.

The smaller fragment is used for the more expensive one-mask-per-position score and gradient calculations.

In [2]:
SEQUENCE = "+MALTKVEKRNRIK+aatttaaggaa-MLGIDNIERVKP"
SHORT_SEQUENCE = "+MALTKV+aatt-MLGIDN"
print(SEQUENCE)

+MALTKVEKRNRIK+aatttaaggaa-MLGIDNIERVKP


## Embeddings and token labels

The pooled vector averages all unpadded tokens, including strand markers. Optional raw logits have one row per token and 24 columns in `ACDEFGHIKLMNPQRSTVWYacgt` order.

In [3]:
display_api_reference("glm2-embedding", "input", "run_glm2_embeddings")
display_api_reference("glm2-embedding", "config", "run_glm2_embeddings")
display_api_reference("glm2-embedding", "output", "run_glm2_embeddings")

**Input** — `MixedSequenceInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Prepared loci: uppercase protein, lowercase acgt, +/- strand markers (<+>/<-> accepted) |

**Config** — `GLM2EmbeddingsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device used for model inference |
| <code>model_checkpoint</code> | <code>Literal['tattabio/gLM2_150M', 'tattabio/gLM2_650M']</code> | <code>'tattabio/gLM2_650M'</code> | gLM2 checkpoint; downloaded automatically from Hugging Face |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences or masked PLL variants per forward pass; reduce if memory is limited |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include token-aligned logits over the 24 canonical biological tokens |
| <code>repr_layer</code> | <code>int</code> | <code>-1</code> | 0=embedding table, 1..N=transformer outputs, -1=last transformer output |

**Output** — `MixedEmbeddingsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[MixedSequenceEmbedding]</code> | required | Embedding bundles in input order |

**`MixedSequenceEmbedding`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>mean_embedding</code> | <code>list[float]</code> | required | Mean-pooled embedding vector (averaged over sequence length) |
| <code>attention_mask</code> | <code>list[int]</code> | required | Binary mask: 1 = valid position, 0 = padding |
| <code>logits</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Optional per-token biological logits in vocabulary order |
| <code>projection</code> | <code>Projection2D &#124; None</code> | <code>None</code> | 2D UMAP projection of this sequence's embedding within the call's batch |
| <code>tokens</code> | <code>list[str]</code> | required | Unpadded token axis, including strand markers |
| <code>vocab</code> | <code>list[str]</code> | <code>['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', 'a', 'c', 'g', 't']</code> | Biological-logit column order |

**`Projection2D`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>x</code> | <code>float</code> | required | First reduced coordinate |
| <code>y</code> | <code>float</code> | required | Second reduced coordinate |

In [4]:
embeddings_result = run_glm2_embeddings(
    GLM2EmbeddingsInput(sequences=[SEQUENCE]),
    GLM2EmbeddingsConfig(
        model_checkpoint=CHECKPOINT, device=DEVICE, return_logits=True,
    ),
)
assert embeddings_result.success, embeddings_result.errors
embedding = embeddings_result.results[0]
print("Pooled dimension:", len(embedding.mean_embedding))
print("Token count:", len(embedding.tokens))
print("Logit columns:", embedding.vocab)
print("First tokens:", list(enumerate(embedding.tokens[:8], start=1)))

Starting worker


Running glm2


Loading tattabio/gLM2_150M on cuda:0


Pooled dimension: 640
Token count: 39
Logit columns: ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', 'a', 'c', 'g', 't']
First tokens: [(1, '+'), (2, 'M'), (3, 'A'), (4, 'L'), (5, 'T'), (6, 'K'), (7, 'V'), (8, 'E')]


## Masked pseudo-log-likelihood

Every canonical protein/DNA position is masked in turn and scored from its context. Markers and ambiguous protein context are excluded from the score targets. Higher mean log-likelihood and lower perplexity indicate greater model predictability; they do not establish function.

In [5]:
display_api_reference("glm2-score", "input", "run_glm2_score")
display_api_reference("glm2-score", "config", "run_glm2_score")
display_api_reference("glm2-score", "output", "run_glm2_score")

**Input** — `MixedSequenceInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Prepared loci: uppercase protein, lowercase acgt, +/- strand markers (<+>/<-> accepted) |

**Config** — `GLM2ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device used for model inference |
| <code>model_checkpoint</code> | <code>Literal['tattabio/gLM2_150M', 'tattabio/gLM2_650M']</code> | <code>'tattabio/gLM2_650M'</code> | gLM2 checkpoint; downloaded automatically from Hugging Face |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences or masked PLL variants per forward pass; reduce if memory is limited |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include masked biological logits; unscored context rows contain zeros |

**Output** — `MixedScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>scores</code> | <code>list[MixedScoringMetrics]</code> | required | Per-locus scores in input order |

**Metrics** — `MixedScoringMetrics` (one set per `scores` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  | Sum over all positions. Grows with sequence length, so compare only at equal length. |
| <code>avg_log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  | Mean per position. Length independent, so comparable across sequences. |
| <code>perplexity</code> **(primary)** | <code>float</code> | <code>&gt;= 1</code> |  | Exponential of the negated mean log-likelihood. Lower means a more confident model. |

In [6]:
score_result = run_glm2_score(
    GLM2ScoringInput(sequences=[SHORT_SEQUENCE]),
    GLM2ScoringConfig(
        model_checkpoint=CHECKPOINT, device=DEVICE, batch_size=1, return_logits=True,
    ),
)
assert score_result.success, score_result.errors
score = score_result.scores[0]
print("Mean log-likelihood:", score["avg_log_likelihood"])
print("Perplexity:", score["perplexity"])
print("Scored token positions:", score.scored_positions)

Starting worker


Running glm2


Loading tattabio/gLM2_150M on cuda:0


Mean log-likelihood: -2.4823607243597507
Perplexity: 11.969487752295684
Scored token positions: [2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19]


## Sampling while preserving modality

The automatic strategy selects two canonical biological positions. `fixed_positions=[2]` protects the first amino acid because the initial strand marker occupies position 1. The model may sample the original token, so two selected sites need not mean two substitutions.

In [7]:
display_api_reference("glm2-sample", "input", "run_glm2_sample")
display_api_reference("glm2-sample", "config", "run_glm2_sample")
display_api_reference("glm2-sample", "output", "run_glm2_sample")

**Input** — `MixedSequenceSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Prepared mixed loci; '_' marks editable sites with explicit mask_modalities |
| <code>mask_modalities</code> | <code>list[dict[int, Literal['protein', 'dna']]] &#124; None</code> | <code>None</code> | Per-locus maps from 1-indexed masked token positions to protein or dna |

**Config** — `GLM2SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device used for model inference |
| <code>model_checkpoint</code> | <code>Literal['tattabio/gLM2_150M', 'tattabio/gLM2_650M']</code> | <code>'tattabio/gLM2_650M'</code> | gLM2 checkpoint; downloaded automatically from Hugging Face |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences or masked PLL variants per forward pass; reduce if memory is limited |
| <code>masking_strategy</code> | <code>MaskingStrategy</code> | <code>MaskingStrategy(method='random', num_mutations=None, mask_fraction=None, fixed_positions=None, temperature=1.0)</code> | Select editable model-token positions; explicit '_' masks bypass selection |
| <code>sampling_method</code> | <code>Literal['single_pass', 'iterative_refinement']</code> | <code>'single_pass'</code> | Fill all masks once, or progressively commit predictions across refinement rounds |
| <code>temperature</code> | <code>float</code> | <code>1.0</code> | Sampling temperature within each position's protein or DNA alphabet |
| <code>top_p</code> | <code>float</code> | <code>1.0</code> | Iterative nucleus threshold; 1.0 disables nucleus filtering |
| <code>num_steps</code> | <code>int</code> | <code>20</code> | Number of iterative refinement rounds |
| <code>schedule</code> | <code>Literal['cosine', 'linear']</code> | <code>'cosine'</code> | Iterative unmask schedule |
| <code>strategy</code> | <code>Literal['random', 'entropy']</code> | <code>'random'</code> | Commit masked sites randomly or by lowest predictive entropy |
| <code>temperature_annealing</code> | <code>bool</code> | <code>True</code> | Cool sampling temperature over iterative refinement rounds |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include biological logits from a final forward pass over each completed sequence |

**Output** — `MixedSampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[MixedSequenceSample]</code> | required | Sampled locus bundles in input order |

**`MixedSequenceSample`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | Completed mixed protein/DNA locus with strand markers |
| <code>logits</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Optional per-token biological logits with shape (L, 24) |
| <code>tokens</code> | <code>list[str]</code> | required | Completed model-token sequence, including strand markers |
| <code>vocab</code> | <code>list[str]</code> | <code>['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', 'a', 'c', 'g', 't']</code> | Biological-logit column order |

In [8]:
sample_result = run_glm2_sample(
    GLM2SampleInput(sequences=[SEQUENCE]),
    GLM2SampleConfig(
        model_checkpoint=CHECKPOINT,
        device=DEVICE,
        masking_strategy=MaskingStrategy(num_mutations=2, fixed_positions=[2]),
        temperature=1.0,
        seed=7,
    ),
)
assert sample_result.success, sample_result.errors
sample = sample_result.results[0]
assert len(sample.tokens) == len(embedding.tokens)
assert sample.tokens[1] == embedding.tokens[1]
print("Original:", SEQUENCE)
print("Sampled: ", sample.sequence)

Starting worker


Running glm2


Loading tattabio/gLM2_150M on cuda:0


Original: +MALTKVEKRNRIK+aatttaaggaa-MLGIDNIERVKP
Sampled:  +MALRKVEKRNRIK+aatttaaggaa-MLGIINIERVKP


### Explicit masks with a protein/DNA label per site

An underscore does not carry case, so each premasked position needs an explicit modality. The mapping uses 1-indexed token positions. This example fills one protein and one DNA site through four refinement rounds. Explicit masks bypass automatic position selection.

In [9]:
masked_tokens = list(embedding.tokens)
protein_position = 2
dna_position = next(i for i, token in enumerate(masked_tokens, start=1) if token in "acgt")
masked_tokens[protein_position - 1] = "_"
masked_tokens[dna_position - 1] = "_"

premasked_result = run_glm2_sample(
    GLM2SampleInput(
        sequences=["".join(masked_tokens)],
        mask_modalities=[{protein_position: "protein", dna_position: "dna"}],
    ),
    GLM2SampleConfig(
        model_checkpoint=CHECKPOINT,
        device=DEVICE,
        sampling_method="iterative_refinement",
        num_steps=4,
        top_p=0.9,
        temperature_annealing=False,
        seed=7,
    ),
)
assert premasked_result.success, premasked_result.errors
completed = premasked_result.results[0]
assert completed.tokens[protein_position - 1] in "ACDEFGHIKLMNPQRSTVWY"
assert completed.tokens[dna_position - 1] in "acgt"
print(completed.sequence)

Starting worker


Running glm2


Loading tattabio/gLM2_150M on cuda:0


+IALTKVEKRNRIK+aatttaaggaa-MLGIDNIERVKP


## Relaxed-sequence gradient

The template fixes modality and strand markers. `one_hot_mixed_logits()` creates one 24-column row per token, with zero rows for fixed context. The default input temperature applies softmax within the allowed alphabet; use `temperature=None` only for already normalized modality-restricted probabilities. The model weights remain frozen.

In [10]:
display_api_reference("glm2-gradient", "input", "run_glm2_gradient")
display_api_reference("glm2-gradient", "config", "run_glm2_gradient")
display_api_reference("glm2-gradient", "output", "run_glm2_gradient")

**Input** — `MixedSequenceGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | Prepared template fixing each token's modality and strand markers |
| <code>logits</code> | <code>list[list[float]]</code> | required | Token-aligned (L, 24) state in ACDEFGHIKLMNPQRSTVWYacgt order; fixed rows are zero |
| <code>temperature</code> | <code>float &#124; None</code> | <code>1.0</code> | Apply softmax(input / T); None requires modality-restricted probabilities |

**Config** — `GLM2GradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device used for model inference |
| <code>model_checkpoint</code> | <code>Literal['tattabio/gLM2_150M', 'tattabio/gLM2_650M']</code> | <code>'tattabio/gLM2_650M'</code> | gLM2 checkpoint; downloaded automatically from Hugging Face |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences or masked PLL variants per forward pass; reduce if memory is limited |
| <code>use_ste</code> | <code>bool</code> | <code>False</code> | Use hard one-hot forward tokens with soft-probability gradients |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward and return the gradient; False evaluates only the masked PLL objective |

**Output** — `MixedGradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>gradient</code> | <code>list[list[float]] &#124; list[list[list[float]]] &#124; None</code> | <code>None</code> | Gradient of the loss with respect to the input logits; matches input shape and column order |
| <code>loss</code> | <code>float</code> | required | Scalar objective for this sequence; lower is better (typically a positive negative log-likelihood) |
| <code>metrics</code> | <code>dict[str, Any]</code> | <code>{}</code> | Auxiliary metrics reported alongside the scalar objective value. |
| <code>vocab</code> | <code>list[str]</code> | required | Symbols defining the column ordering for both the input logits and the returned gradient. |
| <code>tokens</code> | <code>list[str]</code> | required | Model-token axis shared by input logits and returned gradient |

In [11]:
gradient_result = run_glm2_gradient(
    GLM2GradientInput(
        sequence=SHORT_SEQUENCE,
        logits=one_hot_mixed_logits(SHORT_SEQUENCE, sharpness=2.0),
        temperature=1.0,
    ),
    GLM2GradientConfig(model_checkpoint=CHECKPOINT, device=DEVICE, batch_size=1),
)
assert gradient_result.success, gradient_result.errors
assert gradient_result.gradient is not None
print("Masked mean NLL:", gradient_result.loss)
print("Gradient shape:", (len(gradient_result.gradient), len(gradient_result.vocab)))
for token, row in zip(gradient_result.tokens, gradient_result.gradient, strict=True):
    if token in ("+", "-"):
        assert all(value == 0.0 for value in row)

Starting worker


Running glm2


Loading tattabio/gLM2_150M on cuda:0


Masked mean NLL: 2.2869770377874374
Gradient shape: (19, 24)


## Export

Exports go to the temporary directory created above. Embedding CSV contains pooled vectors; score, sample, and gradient JSON preserve their result metadata.

In [12]:
embeddings_result.export(name="glm2_embeddings", export_path=OUTPUT_DIR, file_format="csv")
score_result.export(name="glm2_scores", export_path=OUTPUT_DIR, file_format="json")
sample_result.export(name="glm2_samples", export_path=OUTPUT_DIR, file_format="json")
gradient_result.export(name="glm2_gradient", export_path=OUTPUT_DIR, file_format="json")
print("Written files:", sorted(path.name for path in OUTPUT_DIR.iterdir()))

Written files: ['glm2_embeddings.csv', 'glm2_gradient.json', 'glm2_samples.json', 'glm2_scores.json']
